In [1]:
from pathlib import Path

from lassi.profiler import Timer, MultiProfiler, GPUProfiler
from lassi.source_file import SourceFile
from lassi.executer import FunctionalValidator
from typing import Annotated


from openai import AsyncOpenAI
from groq import Groq


from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

## Minimal C Example

In [ ]:
# Declare a new file

source_file = SourceFile(
    file_name = Path("test.c")
)

source_file

In [ ]:
source_file.write( # this could be the LLM output
    """#include <stdio.h>

        int main(void) {
            printf("Compiler test successful!");
            return 0;
        }
    """
)

executable = source_file.compile()
success = source_file.execute()
report = source_file.get_execution_report()

In [ ]:
report # default report is latency

## More complex CUDA example

In [ ]:
source_file = SourceFile(
    file_name = Path("similarity_cuda_test.cu"),
    folder_path = Path("/home/gbrun/test_cuda_folder/src/") # <-- from another folder
)

In [ ]:
source_file # this time lang is CUDA

In [ ]:
source_file.compile(
    kwds="-O3", # all flags go here
    output_file=Path("test_cuda") # custom name
    ) 

In [ ]:
source_file.execute(
    args="100 100", # need args
    profiler=GPUProfiler() # GPU profiler. There is NVIDIA and AMD
    )

In [ ]:
source_file.get_execution_report() # more complex report

## Code Generation

In [2]:
import dotenv
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

from openai import AsyncOpenAI

In [3]:
dotenv.load_dotenv()

True

In [4]:
source_file = SourceFile(
    file_name = Path("llm_generated_code.c"),
)

In [5]:
@dataclass
class CodeGenDependencies:
    language: Annotated[str, Field(description="Language to implement the code in.")]

@dataclass
class CodeGenOutput:
    code: Annotated[str, Field(description="Valid code that follows the requested task. Code only.")]

In [6]:
MODEL_NAME = "openai/gpt-oss-120b"

### Groq model

In [7]:
model = GroqModel(MODEL_NAME)

### Locally hosted OpenAI (vLLM)

In [ ]:
# model = OpenAIChatModel(MODEL_NAME, provider=OpenAIProvider(base_url="http://localhost:8000/v1"))

### Declare Agent

In [8]:
agent = Agent(
    model=model,
    instructions=(
        "You are a helpful coding tool."
    ),
    output_type= CodeGenOutput,
)

### Create and test a new file

In [ ]:
deps = CodeGenDependencies(language=source_file.lang.value)

result = await agent.run("Write me a C program that takes from args a number N and prints the first N args of the fibonacci sequence.")

source_file.write(result.output.code)

In [ ]:
print(source_file.read())

In [9]:
source_file.compile()

unit_test = [
    FunctionalValidator(
        args = "4",
        golden_output="0 1 1 2\n",
        ret_code=0),
    FunctionalValidator(
        args = "1",
        golden_output="0\n",
        ret_code=0),
    FunctionalValidator(
        args = "5",
        golden_output="0 1 1 2 3\n",
        ret_code=0),
]

output = source_file.execute(
    validator = unit_test
)

source_file.get_execution_history()

Compiling /home/gbrun/LASSI-TOOLS/llm_generated_code.c using gcc...
Compiling with command: gcc /home/gbrun/LASSI-TOOLS/llm_generated_code.c -o /home/gbrun/LASSI-TOOLS/llm_generated_code
Running with command: /home/gbrun/LASSI-TOOLS/llm_generated_code 4
Validating output
Running with command: /home/gbrun/LASSI-TOOLS/llm_generated_code 1
Validating output
Running with command: /home/gbrun/LASSI-TOOLS/llm_generated_code 5
Validating output
Passed: 3 / 3


[TimerReport(latency=0.0013000830076634884),
 TimerReport(latency=0.0015598032623529434),
 TimerReport(latency=0.0016670441254973412)]

In [ ]:
output

In [ ]:
source_file